[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jumafernandez/doctorado-unsl/blob/main/packages/contextual-turn-embeddings/training/contextual-turn-encoder-base/05_train_contextual_v3_colab.ipynb)

# `contextual-turn-encoder-base` **v3** — entrenamiento en M2

**Gemela GPU (Colab) de la [`04`](04_train_contextual_v3_m2.ipynb).** Mismo **v3** (BERT-base literal 12/12/3072 + receta de BERT) pero en **GPU** → horas en vez de días. Lee los datos desde **Google Drive** (los que dejó la nb01).

> ⚠️ **Antes de correr:** (1) subí a Drive `dialogs-2.0.pkl` (de `ANN-UNSL/data/`) junto a `dialogs-full.pkl` y `base_embeddings.npy` en `…/d2f-full/`; (2) activá GPU en *Runtime > Change runtime type > GPU (T4/A100)*.

> Tres modelos, ningún registro se pisa:
> - **v1** — arquitectura custom (pre-LN + residual), receta propia (`lr 2e-4`). *(notebook 02)*
> - **v2** — arquitectura **BERT-fiel** (post-LN), **receta del v1** (`lr 2e-4`, 6 capas/8 heads). *(notebook 03)*
> - **v3** — **BERT-base literal** (12 capas/12 heads), **receta de BERT** (`lr 1e-4`, no-decay en bias/LayerNorm). *(notebook 04)*
>
> En `config.json`, `arch` distingue solo **arquitectura** → v1=`"v1"`, **v2 y v3 son `"v2"`** (misma estructura
> de módulos); lo que los separa es **tamaño + receta**, anotado en `trainlog.jsonl` (`tag`, `recipe`).
> Divergencias vs BERT en [`docs/model/v2.md`](../../docs/model/v2.md).

## 1. Instalar el paquete (clona el repo público — sin token, sin intervención)

In [ ]:
# Instala el paquete desde el repo (PÚBLICO) — sin token, corre solo.
import os, sys, subprocess
REPO = "/content/doctorado-unsl"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jumafernandez/doctorado-unsl.git", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                f"{REPO}/packages/contextual-turn-embeddings"], check=True)
print("paquete instalado desde", REPO)

## 2. Montar Drive, rutas e identidad de la corrida (GPU)

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

REPO = "/content/doctorado-unsl"
RECIPE = os.path.join(REPO, "packages/contextual-turn-embeddings/training/contextual-turn-encoder-base")
CONFIG_PATH = os.path.join(RECIPE, "config.yaml")
sys.path.insert(0, RECIPE)                       # heldout.py

# >>> Drive montado; ruta confirmada de tu MyDrive (Doctorado/contextual-turn-encoder-base) <<<
import shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/Doctorado/contextual-turn-encoder-base"
SRC = os.path.join(DRIVE, "d2f-full")
assert os.path.isdir(SRC), f"No existe {SRC} — ajustá DRIVE a la ruta real en tu MyDrive."

# El memmap grande se COPIA a SSD local (acceso random rápido en el train); los .pkl se leen de Drive.
LOCAL = "/content/d2f-data"; os.makedirs(LOCAL, exist_ok=True)
BASE_EMB_PATH = os.path.join(LOCAL, "base_embeddings.npy")
if not os.path.exists(BASE_EMB_PATH):
    print("copiando base_embeddings.npy a SSD local (~6 GB, unos minutos)...")
    shutil.copy(os.path.join(SRC, "base_embeddings.npy"), BASE_EMB_PATH)
META_PATH = os.path.join(SRC, "dialogs-full.pkl")          # se lee 1 vez a RAM (chico)
HELDOUT_META = os.path.join(SRC, "dialogs-2.0.pkl")
OUT = os.path.join(DRIVE, "models")                        # checkpoints -> Drive: PERSISTEN al runtime
os.makedirs(OUT, exist_ok=True)
print("memmap local:", BASE_EMB_PATH, "| checkpoints -> Drive:", OUT)

# identidad de ESTA corrida (v3 = BERT-base literal + receta de BERT)
VER = "v3"; CORPUS = "full"
LR = 1e-4; EPOCHS = 30                            # schedule BERT completo (en GPU sale barato; best/ cae antes igual)
NLAYERS_OVERRIDE = 12; NHEADS_OVERRIDE = 12; BERT_RECIPE = True
DEVICE = "cuda"

from contextual_turn_embeddings import (Config, DialogueDataset, EmbeddingRetrievalConfig,
    build_model, collate_dialogues, compute_objectives, get_device, resolve_losses_for_mode, set_seed)
from contextual_turn_embeddings.train import build_linear_warmup_scheduler
import heldout as H
assert torch.cuda.is_available(), "No hay GPU: Runtime > Change runtime type > GPU (T4/A100)."
for p in (META_PATH, BASE_EMB_PATH, HELDOUT_META):
    assert os.path.exists(p), f"Falta: {p}"
print("device:", get_device(DEVICE), "| VER:", VER, "| LR:", LR, "| capas/heads:",
      NLAYERS_OVERRIDE, "/", NHEADS_OVERRIDE, "| EPOCHS:", EPOCHS)

## 3. Cargar datos (memmap desde Drive) y excluir held-out

In [ ]:
df = pd.read_pickle(META_PATH)[["dialogue_id", "turn_id", "speaker", "utterance"]].copy()
df["row_id"] = np.arange(len(df), dtype=np.int64)          # = índice en el memmap

emb = np.load(BASE_EMB_PATH, mmap_mode="r")                # NO entra en RAM
assert len(df) == len(emb), (len(df), len(emb))

df_1m = pd.read_pickle(HELDOUT_META)
heldout_ids = H.heldout_dialogue_ids(df_1m)
train_mask, _ = H.split_train_heldout(df, heldout_ids)
df_train = df[train_mask].reset_index(drop=True)

print(f"corpus {CORPUS}: {len(df)} turnos / {df.dialogue_id.nunique()} diálogos")
print(f"held-out: {len(heldout_ids)} diálogos | train: {len(df_train)} turnos / "
      f"{df_train.dialogue_id.nunique()} diálogos")

## 4. Función de entrenamiento (memmap, por modo)

In [ ]:
def train_variant(df_train, emb_memmap, base_cfg, mode, num_layers, num_heads, out_dir,
                  tag, bert_recipe, retrieval=True):
    set_seed(base_cfg.training.seed)
    device = get_device(base_cfg.training.device)

    rng = np.random.default_rng(123)                       # val chica reproducible (curva / best ckpt)
    dids = pd.unique(df_train["dialogue_id"])
    val_dids = set(rng.choice(dids, size=max(1, int(len(dids) * 0.02)), replace=False))
    is_val = df_train["dialogue_id"].isin(val_dids).to_numpy()
    df_tr, df_va = df_train[~is_val], df_train[is_val]

    d = base_cfg.to_dict()
    d["model"]["arch"] = "v2"                               # v2 y v3 comparten arquitectura BERT-fiel
    cfg = Config.from_dict(d)
    cfg.model.attention_mode = mode
    cfg.model.num_layers = num_layers
    cfg.model.num_heads = num_heads
    D = int(emb_memmap.shape[1])
    cfg.model.input_dim = cfg.model.hidden_dim = cfg.model.output_dim = D
    cfg.model.ff_dim = 4 * D                               # 3072 = 4*hidden (intermediate BERT-base)

    losses = resolve_losses_for_mode(cfg.losses, mode)
    if retrieval:
        losses.embedding_retrieval = EmbeddingRetrievalConfig(
            enabled=True, target=("masked" if mode == "bidirectional" else "next_turn"))

    mk = lambda dd: DialogueDataset(dd, emb_memmap, max_turns=cfg.data.max_turns,
                                    num_speakers=cfg.model.num_speakers, lazy=True)
    tr_loader = DataLoader(mk(df_tr), batch_size=cfg.training.batch_size, shuffle=True,
                           num_workers=0, collate_fn=collate_dialogues)
    va_loader = DataLoader(mk(df_va), batch_size=cfg.training.batch_size, shuffle=False,
                           num_workers=0, collate_fn=collate_dialogues)

    model = build_model(cfg.model).to(device)
    if bert_recipe:                                        # receta de BERT
        no_decay = ("bias", "LayerNorm.weight")
        grouped = [
            {"params": [p for n, p in model.named_parameters()
                        if p.requires_grad and not any(nd in n for nd in no_decay)],
             "weight_decay": cfg.training.weight_decay},
            {"params": [p for n, p in model.named_parameters()
                        if p.requires_grad and any(nd in n for nd in no_decay)],
             "weight_decay": 0.0},
        ]
        opt = torch.optim.AdamW(grouped, lr=cfg.training.learning_rate, eps=1e-6)
    else:                                                  # receta del v1 (AdamW uniforme)
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.training.learning_rate,
                                weight_decay=cfg.training.weight_decay)
    total = max(1, len(tr_loader) * cfg.training.epochs)
    sched = build_linear_warmup_scheduler(opt, int(total * cfg.training.warmup_ratio), total)
    nparams = sum(p.numel() for p in model.parameters())
    print(f"[{tag}/{mode}] {type(model).__name__} L={num_layers} A={num_heads} | "
          f"{nparams/1e6:.1f}M params | recipe={'bert' if bert_recipe else 'v1'}", flush=True)

    def move(b):
        o = dict(b)
        o["embeddings"] = b["embeddings"].to(device)
        o["attention_mask"] = b["attention_mask"].to(device)
        if b.get("speaker_ids") is not None:
            o["speaker_ids"] = b["speaker_ids"].to(device)
        return o

    @torch.no_grad()
    def validate():
        model.eval(); set_seed(999); tot = n = 0
        for b in va_loader:
            b = move(b); out = compute_objectives(model, b, losses)
            bs = b["embeddings"].shape[0]; tot += float(out["total"].detach().cpu()) * bs; n += bs
        model.train(); return tot / max(1, n)

    os.makedirs(out_dir, exist_ok=True)
    logf = open(os.path.join(out_dir, "trainlog.jsonl"), "w"); best = float("inf"); t0 = time.time()
    for ep in range(1, cfg.training.epochs + 1):
        model.train(); run = 0.0; te = time.time()
        for i, b in enumerate(tr_loader):
            b = move(b); opt.zero_grad(set_to_none=True)
            loss = compute_objectives(model, b, losses)["total"]; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_norm)
            opt.step(); sched.step(); run += float(loss.detach().cpu())
            if i % cfg.training.log_interval == 0:
                print(f"[{tag}/{mode}] ep{ep} {i}/{len(tr_loader)} loss={float(loss.detach().cpu()):.4f}", flush=True)
        rec = {"tag": tag, "arch": "v2", "recipe": ("bert" if bert_recipe else "v1"),
               "n_layers": num_layers, "n_heads": num_heads, "lr": cfg.training.learning_rate,
               "mode": mode, "epoch": ep, "train_loss": round(run / max(1, len(tr_loader)), 5),
               "val_loss": round(validate(), 5), "epoch_sec": round(time.time() - te, 1)}
        print("EPOCH", json.dumps(rec), flush=True); logf.write(json.dumps(rec) + "\n"); logf.flush()
        model.save_pretrained(out_dir, training_args=rec)
        if rec["val_loss"] < best:
            best = rec["val_loss"]; model.save_pretrained(os.path.join(out_dir, "best"), training_args=rec)
    cfg.to_yaml(os.path.join(out_dir, "config.yaml"))
    print(f"[{tag}/{mode}] DONE best_val={best:.5f} min={(time.time()-t0)/60:.1f}", flush=True)
    return model

## 5. Entrenar v3 en GPU — **una tanda por modo**

Misma receta que la 04 (`lr=1e-4`, AdamW no-decay en bias/LayerNorm, `eps=1e-6`, warmup + decay lineal), pero en `cuda` y **`EPOCHS=30`** (el schedule BERT completo; en GPU sale barato). Corré **5a (AR)** y, cuando termine, **5b (Bidi)**.

> El `best/` cae antes de la 30 igual (la curva converge ~ep11-15), así que no perdés nada: las 30 corren el schedule fiel y el mejor checkpoint queda guardado en **Drive**.

### 5a. AR (autoregresivo) — corré esta PRIMERO

In [ ]:
# Corré PRIMERO esta (AR). Checkpoints en Drive -> sobreviven si se corta el runtime.
base_cfg = Config.from_yaml(CONFIG_PATH)
base_cfg.training.epochs = EPOCHS
base_cfg.training.learning_rate = LR
base_cfg.training.device = DEVICE                # GPU (config.yaml trae mps)
NAME = "contextual-turn-encoder-base"
out_ar = os.path.join(OUT, f"{NAME}-{VER}-ar-{CORPUS}")
assert not os.path.exists(os.path.join(out_ar, "trainlog.jsonl")) or os.environ.get("OVERWRITE"), \
    f"Ya existe {out_ar} — borralo o exportá OVERWRITE=1. (Protege los registros.)"
m_ar = train_variant(df_train, emb, base_cfg, "autoregressive", NLAYERS_OVERRIDE, NHEADS_OVERRIDE,
                     out_ar, tag=VER, bert_recipe=BERT_RECIPE)
print("AR listo ->", out_ar)

### 5b. Bidi (full-context) — corré esta DESPUÉS del AR

In [ ]:
# Corré esta DESPUÉS del AR (si se reinició el runtime, re-corré install + celdas de arriba).
base_cfg = Config.from_yaml(CONFIG_PATH)
base_cfg.training.epochs = EPOCHS
base_cfg.training.learning_rate = LR
base_cfg.training.device = DEVICE
NAME = "contextual-turn-encoder-base"
out_bidi = os.path.join(OUT, f"{NAME}-{VER}-bidi-{CORPUS}")
assert not os.path.exists(os.path.join(out_bidi, "trainlog.jsonl")) or os.environ.get("OVERWRITE"), \
    f"Ya existe {out_bidi} — borralo o exportá OVERWRITE=1. (Protege los registros.)"
m_bidi = train_variant(df_train, emb, base_cfg, "bidirectional", NLAYERS_OVERRIDE, NHEADS_OVERRIDE,
                       out_bidi, tag=VER, bert_recipe=BERT_RECIPE)
print("Bidi listo ->", out_bidi)

## 6. Después

Los checkpoints quedaron en **Drive** (persisten aunque se corte el runtime):
`…/Doctorado/contextual-turn-encoder-base/models/…-v3-{ar,bidi}-full/best/`.

- Bajalos a tu repo local (`packages/contextual-turn-embeddings/models/`) y corré
  `python plot_full_results.py` → suma **v3** solo (v1/v2/v3).
- Downstream (act-match / MSS@10): `packages/conversational-ann/scripts/eval_prelim.py`.
- Si Colab se desconecta a mitad, el `best/` ya está en Drive: re-corré install + setup + la celda del modo
  que falte (no vuelve a copiar el memmap si ya está en `/content/d2f-data/`).